In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import time
import pandas as pd
import json
import logging
import ast
from scipy.stats import pearsonr
from collections import deque
from torch.utils.data import Dataset
from query2Vec.util import Normalizer
from query2Vec.database_util import collator, Encoder, TreeNode
from query2Vec.dataset import PlanTreeDataset
from query2Vec.model import QueryFormer

In [ ]:
class Args:
    # bs = 1024
    # SQ: smaller batch size
    bs = 128
    lr = 0.001
    # epochs = 200
    epochs = 20
    clip_size = 50
    embed_size = 64
    pred_hid = 128
    ffn_dim = 128
    head_size = 12
    n_layers = 8
    dropout = 0.1
    sch_decay = 0.6
    device = 'cuda:0'
    newpath = './results/full/cost/'
    to_predict = 'cost'
  
args = Args()
if not os.path.exists(args.newpath):
    os.makedirs(args.newpath)

In [ ]:
from query2Vec.util import seed_everything
seed_everything()

# init encoder
col2idx = {
    "NA": 0,
    "u_user_id": 1,
    "u_gender": 2,
    "u_age": 3,
    "u_occupation": 4,
    "u_zipcode": 5,
    "u_features": 6,
    "m_movie_id": 7,
    "m_title": 8,
    "m_genres": 9,
    "m_spoken_languages": 10,
    "m_popularity": 11,
    "m_vote_average": 12,
    "m_vote_count": 13,
    "m_features": 14,
    "mt_movie_id": 15,
    "mt_relevance_score": 16,
}
encoder = Encoder({}, {}, col2idx)

In [ ]:
to_predict = 'cost'
dir_path = "/home/velox/velox/optimizer/tests/generatedQueryPlan"
df = pd.read_csv(os.path.join(dir_path, "model_benchmark_results_YOURTIMESTAMP.csv"), sep="|")
df = df[df["error"].isna()]

max_latency = np.max(df["executionTime"])
cost_norm = Normalizer(0, max_latency)

In [ ]:
from model2Vec.database_util import collator, ModelGraphEncoder, WeisfeilerLehmanEncoder
from model2Vec.dataset import ModelGraphTreeNode, ModelComputationGraphDataset
from model2Vec.model import Model2Vec

# load pre-trained model
cost_norm = torch.load('./cactusdb/model2vec/cost_norm.pt')
model2vec = torch.load('./cactusdb/model2vec/model2vec.pt')
model2vec_cost_norm = torch.load('./cactusdb/model2vec/cost_norm.pt')
wl_encoder = torch.load('./cactusdb/model2vec/wl_encoder.pt')
model_graph_encoder = torch.load('./cactusdb/model2vec/model_graph_encoder.pt')

In [ ]:
import query2Vec.dataset as q2vDataset

query_stats_df = q2vDataset.read_and_process_histograms("/home/velox/velox/optimizer/tests/tableStats.txt")
encoder_min_max_map = q2vDataset.get_numerical_min_max_mapping(query_stats_df)
encoder_cate_map = q2vDataset.get_categorical_mapping(query_stats_df)
encoder.set_column_normalizer(encoder_min_max_map, encoder_cate_map)

In [ ]:
from sklearn.model_selection import train_test_split
from query2Vec.database_util import WeisfeilerLehmanQueryEncoder

import warnings
import logging
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.ERROR)

# Split the dataframe into 80% training and 20% testing
train_df, test_df = train_test_split(df, test_size=0.05, random_state=42)
data_dir = "/home/velox/velox/optimizer/tests"

train_ds = PlanTreeDataset(
    train_df, encoder, to_predict, table_sample=None, cost_norm=cost_norm, dir_path=data_dir,
    model2vec=model2vec, model_graph_encoder=model_graph_encoder,
    model2vec_normalizer=model2vec_cost_norm,
    device=args.device, pre_process_stats=True, query_stats_df=query_stats_df,
)
test_ds = PlanTreeDataset(
    test_df,
    encoder,
    to_predict,
    table_sample=None,
    cost_norm=cost_norm,
    dir_path=data_dir,
    model2vec=model2vec,
    model_graph_encoder=model_graph_encoder,
    model2vec_normalizer=model2vec_cost_norm,
    device=args.device, pre_process_stats=True, query_stats_df=query_stats_df,
)

In [ ]:
wl_query_kernel = WeisfeilerLehmanQueryEncoder(range_percent=0.1, num_iterations=5, embed_sim_threshold=0.5)
wl_query_kernel.obtain_wl_feature_for_dataset(train_ds)
wl_query_kernel.construct_similar_dissimilar_pairs_for_dataset(train_ds)

In [ ]:
from query2Vec.trainer import train_query2vec, train_query2vec_with_contrastive
len(train_ds), len(test_ds)

In [ ]:
args.epochs = 40
args.bs = 32
crit = nn.MSELoss()
model = QueryFormer(
    emb_size=args.embed_size,
    ffn_dim=args.ffn_dim,
    head_size=args.head_size,
    dropout=args.dropout,
    n_layers=args.n_layers,
    use_sample=False,
    use_hist=True,
    pred_hid=args.pred_hid,
)
_ = model.to(args.device)
args.epochs= 20
model, best_path = train_query2vec_with_contrastive(
    model,
    train_ds,
    train_ds,
    crit,
    cost_norm,
    args,
    cost_loss_factor=0,
    contrastive_loss_factor=1,
    log_best=True,
)

In [ ]:
args.epochs= 20
model, best_path = train_query2vec_with_contrastive(
    model,
    train_ds,
    train_ds,
    crit,
    cost_norm,
    args,
    cost_loss_factor=1,
    contrastive_loss_factor=0,
    log_best=True,
)

In [ ]:
crit = nn.MSELoss()
args.epochs = 40
args.bs = 32
model = QueryFormer(
    emb_size=args.embed_size,
    ffn_dim=args.ffn_dim,
    head_size=args.head_size,
    dropout=args.dropout,
    n_layers=args.n_layers,
    use_sample=False,
    use_hist=True,
    pred_hid=args.pred_hid,
)
_ = model.to(args.device)

model, best_path = train_query2vec_with_contrastive(
    model,
    train_ds,
    train_ds,
    crit,
    cost_norm,
    args,
    cost_loss_factor=1,
    contrastive_loss_factor=1,
    log_best=True,
    best_metric="corr"

)